In [5]:
# ============================================================
# GOAT-Net — Phase 3: Spatial Engine (Fixed Setup)
# ============================================================
import os, sys, random
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 1. Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# 2. Mount Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# 3. Set Working Directory (NO GIT PULLING)
PROJECT = "GOAT-Net"
REPO = Path("/content/drive/MyDrive") / PROJECT
os.chdir(REPO)
print(f"📁 Working directory securely set to: {REPO}")

# 4. Fix Python Path
SRC_DIR = REPO / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# 5. Clear cache to prevent ModuleNotFoundError
for m in list(sys.modules):
    if m == "data" or m.startswith("data."):
        del sys.modules[m]

# 6. Install EVERYTHING needed for Phase 3 (including statsbombpy)
print("⏳ Installing dependencies...")
!pip install -q statsbombpy networkx tqdm mplsoccer

# 7. Import Dataset Manager
from data.dataset_manager import load_player_mapping
mapping = load_player_mapping()
statsbomb_to_canonical = dict(zip(mapping["statsbomb_name"], mapping["canonical_id"]))
statsbomb_to_common = dict(zip(mapping["statsbomb_name"], mapping["common_name"]))

# 8. Setup Paths
DATA_ROOT = REPO
PATHS = {
    "repo": REPO,
    "raw": DATA_ROOT / "data" / "raw",
    "processed": DATA_ROOT / "data" / "processed",
    "metadata": DATA_ROOT / "metadata",
    "output_spatial": DATA_ROOT / "data" / "processed" / "spatial",
}
for p in PATHS.values():
    p.mkdir(parents=True, exist_ok=True)

print("✅ Setup complete. Ready for Cell 2!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 Working directory securely set to: /content/drive/MyDrive/GOAT-Net
⏳ Installing dependencies...
✅ Setup complete. Ready for Cell 2!


In [6]:
# ============================================================
# Cell 2: Full Match Event Re‑Pull (for passing networks & all spatial data)
# ============================================================
from statsbombpy import sb

FULL_EVENTS_PATH = PATHS["raw"] / "modern" / "statsbomb" / "statsbomb_full_match_events.parquet"
PANEL_MATCHES_PATH = PATHS["raw"] / "modern" / "statsbomb" / "statsbomb_panel_matches.csv"

if FULL_EVENTS_PATH.exists():
    print("✅ Full events cache found. Loading...")
    full_events = pd.read_parquet(FULL_EVENTS_PATH)
else:
    if not PANEL_MATCHES_PATH.exists():
        raise FileNotFoundError(
            f"{PANEL_MATCHES_PATH} not found. Run 0_data_collection.ipynb through Cell 6 first."
        )
    panel_matches = pd.read_csv(PANEL_MATCHES_PATH)
    match_ids = panel_matches["match_id"].dropna().unique()
    print(f"⬇️ Downloading full events for {len(match_ids)} matches...")

    all_full_events = []
    for mid in tqdm(match_ids, desc="Downloading"):
        try:
            ev = sb.events(match_id=mid)
            ev["match_id"] = mid
            all_full_events.append(ev)
        except Exception as e:
            print(f"  ⚠️ Match {mid} failed: {e}")

    full_events = pd.concat(all_full_events, ignore_index=True) if all_full_events else pd.DataFrame()
    if not full_events.empty:
        full_events.to_parquet(FULL_EVENTS_PATH, index=False)
        print(f"💾 Saved {len(full_events)} events for {full_events['match_id'].nunique()} matches.")
    else:
        print("❌ No events downloaded.")

print(f"Total events loaded: {len(full_events)}")
if not full_events.empty:
    print(f"Event types: {sorted(full_events['type'].dropna().unique())[:20]}")

✅ Full events cache found. Loading...
Total events loaded: 2617850
Event types: ['50/50', 'Bad Behaviour', 'Ball Receipt*', 'Ball Recovery', 'Block', 'Camera On', 'Camera off', 'Carry', 'Clearance', 'Dispossessed', 'Dribble', 'Dribbled Past', 'Duel', 'Error', 'Foul Committed', 'Foul Won', 'Goal Keeper', 'Half End', 'Half Start', 'Injury Stoppage']


In [ ]:
# ============================================================
# GOAT-Net — Phase 3: Spatial Engine (Fixed Setup)
# ============================================================
import os, sys, random
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 1. Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# 2. Mount Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# 3. Set Working Directory (NO GIT PULLING)
PROJECT = "GOAT-Net"
REPO = Path("/content/drive/MyDrive") / PROJECT
os.chdir(REPO)
print(f"📁 Working directory securely set to: {REPO}")

# 4. Fix Python Path
SRC_DIR = REPO / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# 5. Clear cache to prevent ModuleNotFoundError
for m in list(sys.modules):
    if m == "data" or m.startswith("data."):
        del sys.modules[m]

# 6. Install EVERYTHING needed for Phase 3 (including statsbombpy)
print("⏳ Installing dependencies...")
!pip install -q statsbombpy networkx tqdm mplsoccer

# 7. Import Dataset Manager
from data.dataset_manager import load_player_mapping
mapping = load_player_mapping()
statsbomb_to_canonical = dict(zip(mapping["statsbomb_name"], mapping["canonical_id"]))
statsbomb_to_common = dict(zip(mapping["statsbomb_name"], mapping["common_name"]))

# 8. Setup Paths
DATA_ROOT = REPO
PATHS = {
    "repo": REPO,
    "raw": DATA_ROOT / "data" / "raw",
    "processed": DATA_ROOT / "data" / "processed",
    "metadata": DATA_ROOT / "metadata",
    "output_spatial": DATA_ROOT / "data" / "processed" / "spatial",
}
for p in PATHS.values():
    p.mkdir(parents=True, exist_ok=True)

print("✅ Setup complete. Ready for Cell 2!")

# ============================================================
# Cell 2: Full Match Event Re‑Pull (for passing networks & all spatial data)
# ============================================================
from statsbombpy import sb

FULL_EVENTS_PATH = PATHS["raw"] / "modern" / "statsbomb" / "statsbomb_full_match_events.parquet"
PANEL_MATCHES_PATH = PATHS["raw"] / "modern" / "statsbomb" / "statsbomb_panel_matches.csv"

if FULL_EVENTS_PATH.exists():
    print("✅ Full events cache found. Loading...")
    full_events = pd.read_parquet(FULL_EVENTS_PATH)
else:
    if not PANEL_MATCHES_PATH.exists():
        raise FileNotFoundError(
            f"{PANEL_MATCHES_PATH} not found. Run 0_data_collection.ipynb through Cell 6 first."
        )
    panel_matches = pd.read_csv(PANEL_MATCHES_PATH)
    match_ids = panel_matches["match_id"].dropna().unique()
    print(f"⬇️ Downloading full events for {len(match_ids)} matches...")

    all_full_events = []
    for mid in tqdm(match_ids, desc="Downloading"):
        try:
            ev = sb.events(match_id=mid)
            ev["match_id"] = mid
            all_full_events.append(ev)
        except Exception as e:
            print(f"  ⚠️ Match {mid} failed: {e}")

    full_events = pd.concat(all_full_events, ignore_index=True) if all_full_events else pd.DataFrame()
    if not full_events.empty:
        full_events.to_parquet(FULL_EVENTS_PATH, index=False)
        print(f"💾 Saved {len(full_events)} events for {full_events['match_id'].nunique()} matches.")
    else:
        print("❌ No events downloaded.")

print(f"Total events loaded: {len(full_events)}")
if not full_events.empty:
    print(f"Event types: {sorted(full_events['type'].dropna().unique())[:20]}")

# ============================================================
# Cell 3: Coordinate Cleaning & Pitch Zones (Memory Optimized)
# ============================================================
import gc
import numpy as np
import pandas as pd

def unpack_coordinates(df, col_name, prefix):
    """Extract [x, y] using fast vstack, safely ignoring 3D z-coordinates."""
    if col_name not in df.columns: return df
    valid = df[col_name].apply(lambda x: isinstance(x, (list, np.ndarray)) and len(x) >= 2)
    df[f'{prefix}_x'] = np.nan
    df[f'{prefix}_y'] = np.nan
    if valid.any():
        # The [:2] slice ensures we only take X and Y, dropping StatsBomb's Z-axis for high shots
        coords = np.vstack([c[:2] for c in df.loc[valid, col_name]])
        df.loc[valid, f'{prefix}_x'] = coords[:, 0]
        df.loc[valid, f'{prefix}_y'] = coords[:, 1]
    return df

print("⏳ Unpacking coordinates...")
# Avoid full copy, just filter directly to save memory
has_loc = full_events['location'].apply(lambda x: isinstance(x, (list, np.ndarray)) and len(x) >= 2)
events_spatial = full_events[has_loc].reset_index(drop=True)

events_spatial = unpack_coordinates(events_spatial, 'location', 'loc')
events_spatial = unpack_coordinates(events_spatial, 'pass_end_location', 'pass_end')
events_spatial = unpack_coordinates(events_spatial, 'carry_end_location', 'carry_end')
events_spatial = unpack_coordinates(events_spatial, 'shot_end_location', 'shot_end')

print("⏳ Calculating geometric features...")
# 1. Vectorized Distance
GOAL_X, GOAL_Y, HALF_GOAL = 120.0, 40.0, 3.66
events_spatial['dist_to_goal'] = np.sqrt((events_spatial['loc_x'] - GOAL_X)**2 + (events_spatial['loc_y'] - GOAL_Y)**2)

# 2. Vectorized Angle
v1_x = GOAL_X - events_spatial['loc_x']
v1_y = (GOAL_Y - HALF_GOAL) - events_spatial['loc_y']
v2_x = GOAL_X - events_spatial['loc_x']
v2_y = (GOAL_Y + HALF_GOAL) - events_spatial['loc_y']

dot_product = (v1_x * v2_x) + (v1_y * v2_y)
mag_v1 = np.sqrt(v1_x**2 + v1_y**2)
mag_v2 = np.sqrt(v2_x**2 + v2_y**2)
cos_angle = np.clip(dot_product / (mag_v1 * mag_v2 + 1e-10), -1.0, 1.0)
events_spatial['angle_to_goal'] = np.degrees(np.arccos(cos_angle))

# 3. Vectorized Zones
events_spatial['zone'] = np.select(
    [events_spatial['loc_x'] < 40, events_spatial['loc_x'] < 80],
    ['defensive', 'midfield'], default='attacking'
)

in_box_x = events_spatial['loc_x'] >= 102
events_spatial['pen_zone'] = np.select(
    [in_box_x & (events_spatial['loc_y'] >= 18) & (events_spatial['loc_y'] <= 62),
     in_box_x & (events_spatial['loc_y'] < 18),
     in_box_x & (events_spatial['loc_y'] > 62)],
    ['penalty_area', 'left_wing', 'right_wing'], default='outside'
)

# 4. Memory Optimization (Downcast & Categories)
float_cols = events_spatial.select_dtypes(include=['float64']).columns
events_spatial[float_cols] = events_spatial[float_cols].astype('float32')

cat_cols = ['type', 'zone', 'pen_zone', 'competition_name', 'season_name', 'play_pattern']
for c in cat_cols:
    if c in events_spatial.columns:
        events_spatial[c] = events_spatial[c].astype('category')

gc.collect()
print("✅ Coordinates unpacked, zones defined, and RAM optimized.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 Working directory securely set to: /content/drive/MyDrive/GOAT-Net
⏳ Installing dependencies...
✅ Setup complete. Ready for Cell 2!
✅ Full events cache found. Loading...
Total events loaded: 2617850
Event types: ['50/50', 'Bad Behaviour', 'Ball Receipt*', 'Ball Recovery', 'Block', 'Camera On', 'Camera off', 'Carry', 'Clearance', 'Dispossessed', 'Dribble', 'Dribbled Past', 'Duel', 'Error', 'Foul Committed', 'Foul Won', 'Goal Keeper', 'Half End', 'Half Start', 'Injury Stoppage']
⏳ Unpacking coordinates...


In [ ]:
# ============================================================
# Cell 4: Per‑Event Spatial Features (Safe Copies)
# ============================================================
PASS_PROG_THRESH = 30  # Not used now, replaced by distance-to-goal reduction
CARRY_PROG_THRESH = 15  # distance-to-goal reduction threshold (same for both)

# --- Pass features ---
pass_events = events_spatial[events_spatial['type'] == 'Pass'].copy()
if not pass_events.empty:
    # Completion
    if 'pass_outcome' in pass_events.columns:
        pass_events['completed'] = pass_events['pass_outcome'].isna()
    else:
        pass_events['completed'] = True

    # Progressive: reduction in distance to goal ≥ 15
    if 'pass_end_x' in pass_events.columns and 'pass_end_y' in pass_events.columns:
        end_dist_to_goal = np.sqrt((pass_events['pass_end_x'] - GOAL_X)**2 + (pass_events['pass_end_y'] - GOAL_Y)**2)
        pass_events['progressive'] = (pass_events['dist_to_goal'] - end_dist_to_goal) >= 15
    else:
        pass_events['progressive'] = False

    # Key pass
    if 'shot_assist' in pass_events.columns:
        pass_events['key_pass'] = pass_events['shot_assist']
    else:
        pass_events['key_pass'] = False

# --- Shot features ---
shot_events = events_spatial[events_spatial['type'] == 'Shot'].copy()
if not shot_events.empty:
    # Show available outcomes for verification
    print("Shot outcomes found:", sorted(shot_events['shot_outcome'].dropna().unique()))
    if 'shot_outcome' in shot_events.columns:
        shot_events['goal'] = shot_events['shot_outcome'] == 'Goal'
        # FIX: on target = Goal + Saved only (Blocked is NOT on target)
        shot_events['on_target'] = shot_events['shot_outcome'].isin(['Goal', 'Saved'])
    else:
        shot_events['goal'] = False
        shot_events['on_target'] = False
    if 'shot_statsbomb_xg' not in shot_events.columns:
        shot_events['shot_statsbomb_xg'] = np.nan

# --- Carry features ---
carry_events = events_spatial[events_spatial['type'] == 'Carry'].copy()
if not carry_events.empty and 'carry_end_x' in carry_events.columns and 'carry_end_y' in carry_events.columns:
    # Progressive carry: reduction in distance to goal ≥ 15
    end_dist = np.sqrt((carry_events['carry_end_x'] - GOAL_X)**2 + (carry_events['carry_end_y'] - GOAL_Y)**2)
    carry_events['progressive'] = (carry_events['dist_to_goal'] - end_dist) >= 15
else:
    if not carry_events.empty:
        carry_events['progressive'] = False

print("✅ Per‑event features computed (corrected on-target & progressive definitions).")


In [ ]:
# ============================================================
# Cell 5: Build passing graphs per match, compute centrality (Optimized)
# ============================================================
import networkx as nx
import gc
from tqdm.auto import tqdm

CENTRALITY_PATH = PATHS["output_spatial"] / "match_passing_centrality.parquet"

if CENTRALITY_PATH.exists():
    print("✅ Centrality already cached. Loading...")
    centrality_df = pd.read_parquet(CENTRALITY_PATH)
else:
    print("⏳ Extracting passing data to save memory...")

    # --- HUGE RAM SAVIOR ---
    # Filter for passes BEFORE the loop so we don't carry 2.6M rows into the graph builder
    pass_cols = ['match_id', 'player', 'pass_recipient']
    if 'pass_outcome' in full_events.columns:
        pass_cols.append('pass_outcome')

    # Isolate passes and keep only necessary columns
    all_passes = full_events.loc[full_events['type'] == 'Pass', pass_cols]

    # StatsBomb marks completed passes as NaN. Keep only completed passes.
    if 'pass_outcome' in all_passes.columns:
        all_passes = all_passes[all_passes['pass_outcome'].isna()]

    # Drop any rows missing a sender or receiver
    all_passes = all_passes.dropna(subset=['player', 'pass_recipient'])

    def compute_centrality(match_passes, match_id):
        G = nx.DiGraph()
        # Build the graph directly from the pre-filtered rows
        for _, row in match_passes.iterrows():
            src, dst = row['player'], row['pass_recipient']
            G.add_edge(src, dst, weight=G.get_edge_data(src, dst, default={'weight':0})['weight']+1)

        if G.number_of_nodes() == 0:
            return pd.DataFrame()

        between = nx.betweenness_centrality(G, weight='weight', normalized=True)
        close = nx.closeness_centrality(G.reverse())   # who receives easily
        degree = dict(G.degree(weight='weight'))

        rows = [{
            'match_id': match_id,
            'statsbomb_player': p,
            'betweenness': between.get(p, 0),
            'closeness': close.get(p, 0),
            'weighted_degree': degree.get(p, 0)
        } for p in G.nodes()]

        return pd.DataFrame(rows)

    print("⚙️ Computing passing centrality per match...")
    results = []

    # Now we loop over the tiny, lightweight all_passes dataframe
    for mid, match_passes in tqdm(all_passes.groupby('match_id'), desc="Matches"):
        res = compute_centrality(match_passes, mid)
        if not res.empty:
            results.append(res)

    centrality_df = pd.concat(results, ignore_index=True) if results else pd.DataFrame()

    if not centrality_df.empty:
        centrality_df['canonical_id'] = centrality_df['statsbomb_player'].map(statsbomb_to_canonical)
        centrality_df['common_name'] = centrality_df['statsbomb_player'].map(statsbomb_to_common)

        # Downcast floats to save space
        centrality_df[['betweenness', 'closeness']] = centrality_df[['betweenness', 'closeness']].astype('float32')
        centrality_df.to_parquet(CENTRALITY_PATH, index=False)
        print(f"💾 Saved centrality for {len(centrality_df)} player‑matches.")

    # Garbage collection
    del all_passes, results
    gc.collect()

print(f"Centrality records: {len(centrality_df)}")

In [ ]:
# ============================================================
# Cell 6: Defensive Spatial Features
# ============================================================
defensive_types = ['Pressure', 'Ball Recovery', 'Block', 'Interception', 'Tackle', 'Clearance',
                   'Foul Committed', 'Duel']
def_events = events_spatial[events_spatial['type'].isin(defensive_types)].copy()

def_agg = (
    def_events.groupby(['match_id', 'player'])
    .agg(
        total_def_actions=('type', 'count'),
        def_third_actions=('zone', lambda x: (x == 'defensive').sum()),
        avg_def_x=('loc_x', 'mean'),
        avg_def_y=('loc_y', 'mean')
    )
    .reset_index()
    .rename(columns={'player': 'statsbomb_player'})
)

# Map to canonical
def_agg['canonical_id'] = def_agg['statsbomb_player'].map(statsbomb_to_canonical)
def_agg['common_name'] = def_agg['statsbomb_player'].map(statsbomb_to_common)

print(f"Defensive actions aggregated for {len(def_agg)} player‑matches.")


In [ ]:
# ============================================================
# Cell 7: Player‑Match Aggregation (Self-Healing & Safe)
# ============================================================
import gc
import numpy as np
import pandas as pd

# --- SELF-HEALING GUARDS (Rebuilds subsets if wiped from RAM, using corrected progressive) ---
if 'pass_events' not in globals():
    print("⚠️ pass_events missing. Rebuilding from events_spatial...")
    PASS_PROG_THRESH = 30
    pass_events = events_spatial[events_spatial['type'] == 'Pass'].copy()
    pass_events['completed'] = pass_events['pass_outcome'].isna() if 'pass_outcome' in pass_events.columns else True
    # Rebuild progressive using distance-to-goal
    if 'pass_end_x' in pass_events.columns and 'pass_end_y' in pass_events.columns:
        end_dist = np.sqrt((pass_events['pass_end_x'] - GOAL_X)**2 + (pass_events['pass_end_y'] - GOAL_Y)**2)
        pass_events['progressive'] = (pass_events['dist_to_goal'] - end_dist) >= 15
    else:
        pass_events['progressive'] = False
    pass_events['key_pass'] = pass_events['shot_assist'] if 'shot_assist' in pass_events.columns else False

if 'shot_events' not in globals():
    print("⚠️ shot_events missing. Rebuilding from events_spatial...")
    shot_events = events_spatial[events_spatial['type'] == 'Shot'].copy()
    shot_events['goal'] = (shot_events['shot_outcome'] == 'Goal') if 'shot_outcome' in shot_events.columns else False
    # Correct on-target: Goal + Saved only
    shot_events['on_target'] = shot_events['shot_outcome'].isin(['Goal', 'Saved']) if 'shot_outcome' in shot_events.columns else False
    if 'shot_statsbomb_xg' not in shot_events.columns:
        shot_events['shot_statsbomb_xg'] = np.nan

if 'carry_events' not in globals():
    print("⚠️ carry_events missing. Rebuilding from events_spatial...")
    CARRY_PROG_THRESH = 15
    carry_events = events_spatial[events_spatial['type'] == 'Carry'].copy()
    if 'carry_end_x' in carry_events.columns and 'carry_end_y' in carry_events.columns:
        end_dist = np.sqrt((carry_events['carry_end_x'] - GOAL_X)**2 + (carry_events['carry_end_y'] - GOAL_Y)**2)
        carry_events['progressive'] = (carry_events['dist_to_goal'] - end_dist) >= 15
    else:
        carry_events['progressive'] = False

if 'def_events' not in globals():
    print("⚠️ def_events missing. Rebuilding from events_spatial...")
    defensive_types = ['Pressure', 'Ball Recovery', 'Block', 'Interception', 'Tackle', 'Clearance', 'Foul Committed', 'Duel']
    def_events = events_spatial[events_spatial['type'].isin(defensive_types)].copy()

if 'def_agg' not in globals():
    print("⚠️ def_agg missing. Rebuilding defensive aggregations...")
    def_agg = (
        def_events.groupby(['match_id', 'player'])
        .agg(
            total_def_actions=('type', 'count'),
            def_third_actions=('zone', lambda x: (x == 'defensive').sum()),
            avg_def_x=('loc_x', 'mean'),
            avg_def_y=('loc_y', 'mean')
        )
        .reset_index()
        .rename(columns={'player': 'statsbomb_player'})
    )
    def_agg['canonical_id'] = def_agg['statsbomb_player'].map(statsbomb_to_canonical)
    def_agg['common_name'] = def_agg['statsbomb_player'].map(statsbomb_to_common)

# --- STANDARD AGGREGATIONS ---
print("⏳ Running passing aggregation...")
pass_agg = pass_events.groupby(['match_id', 'player']).agg(
    total_passes=('type', 'count'),
    completed_passes=('completed', 'sum'),
    progressive_passes=('progressive', 'sum'),
    key_passes=('key_pass', 'sum'),
    passes_attacking_zone=('zone', lambda x: (x == 'attacking').sum()),
    passes_penalty_area=('pen_zone', lambda x: (x == 'penalty_area').sum()),
    avg_pass_distance=('dist_to_goal', 'mean')
).reset_index().rename(columns={'player': 'statsbomb_player'})

print("⏳ Running shooting aggregation...")
shot_agg = shot_events.groupby(['match_id', 'player']).agg(
    total_shots=('type', 'count'),
    goals=('goal', 'sum'),
    shots_on_target=('on_target', 'sum'),
    total_xG=('shot_statsbomb_xg', 'sum'),
    avg_shot_dist=('dist_to_goal', 'mean'),
    avg_shot_angle=('angle_to_goal', 'mean')
).reset_index().rename(columns={'player': 'statsbomb_player'})

print("⏳ Running carry aggregation...")
carry_agg = carry_events.groupby(['match_id', 'player']).agg(
    total_carries=('type', 'count'),
    progressive_carries=('progressive', 'sum')
).reset_index().rename(columns={'player': 'statsbomb_player'})

# Merge everything
print("⏳ Merging spatial feature blocks...")
match_agg = pass_agg.merge(shot_agg, on=['match_id', 'statsbomb_player'], how='outer')
match_agg = match_agg.merge(carry_agg, on=['match_id', 'statsbomb_player'], how='outer')
match_agg = match_agg.merge(def_agg, on=['match_id', 'statsbomb_player'], how='outer')

# Add centrality (betweenness, closeness, weighted_degree)
if 'centrality_df' in globals() and not centrality_df.empty:
    match_agg = match_agg.merge(
        centrality_df[['match_id', 'statsbomb_player', 'betweenness', 'closeness', 'weighted_degree']],
        on=['match_id', 'statsbomb_player'], how='left'
    )

# Map player IDs and names
match_agg['canonical_id'] = match_agg['statsbomb_player'].map(statsbomb_to_canonical)
match_agg['common_name'] = match_agg['statsbomb_player'].map(statsbomb_to_common)

# Fill NaN for numeric columns
for col in match_agg.columns:
    if match_agg[col].dtype in [np.float64, np.int64, np.float32, np.int32]:
        match_agg[col] = match_agg[col].fillna(0)

# Add competition/season info cleanly
if 'competition_name' in full_events.columns:
    comp_info = (
        full_events
        .groupby("match_id")[["competition_name", "season_name"]]
        .first()
        .reset_index()
    )
    match_agg = match_agg.merge(comp_info, on='match_id', how='left')

# --- GARBAGE COLLECTION ---
del pass_events, shot_events, carry_events, def_events
del pass_agg, shot_agg, carry_agg, def_agg
gc.collect()

print(f"✅ Player‑match records successfully built: {len(match_agg)}")


In [ ]:
# ============================================================
# Cell 8: Merge Season Minutes (for context only – NO match per‑90)
# ============================================================
# ============================================================
# Cell 8: Merge Season Minutes (Type-Safe Fix)
# ============================================================
season_path = PATHS["processed"] / "statistical" / "player_season_stats.parquet"
if season_path.exists():
    season_df = pd.read_parquet(season_path)
    season_min = season_df[['canonical_id', 'Season', 'Minutes']].drop_duplicates().copy()

    # Ensure right-side key is explicitly a string
    season_min['Season'] = season_min['Season'].astype(str)

    # Align season name and enforce string type to prevent float64 mismatch
    if 'season_name' in match_agg.columns:
        match_agg['season_name_clean'] = match_agg['season_name'].fillna('').astype(str).str.replace('/', '-')
        match_agg.loc[match_agg['season_name_clean'] == '', 'season_name_clean'] = None
    else:
        match_agg['season_name_clean'] = None

    match_agg['season_name_clean'] = match_agg['season_name_clean'].astype(str)

    match_agg = match_agg.merge(
        season_min,
        left_on=['canonical_id', 'season_name_clean'],
        right_on=['canonical_id', 'Season'],
        how='left'
    )
    match_agg.rename(columns={'Minutes': 'season_total_minutes'}, inplace=True)
    match_agg.drop(columns=['season_name_clean', 'Season'], inplace=True, errors='ignore')
    print("✅ Season minutes merged safely.")
else:
    print("⚠️ Season stats not found.")
    match_agg['season_total_minutes'] = np.nan

# *** IMPORTANT: We do NOT compute match per‑90 here because we lack match‑level minutes ***
print("ℹ️ Match‑level per‑90 rates are not computed; only career‑level per‑90 will be calculated later.")

In [ ]:
# ============================================================
# Cell 9: Aggregate to Career Level (with Correct Rates & Percentages)
# ============================================================
career_agg = match_agg.groupby(['canonical_id', 'common_name']).agg(
    matches=('match_id', 'nunique'),
    # Pass
    career_passes=('total_passes', 'sum'),
    career_progressive_passes=('progressive_passes', 'sum'),
    career_key_passes=('key_passes', 'sum'),
    # Shot
    career_shots=('total_shots', 'sum'),
    career_goals=('goals', 'sum'),
    career_shots_on_target=('shots_on_target', 'sum'),
    career_xG=('total_xG', 'sum'),
    avg_shot_dist=('avg_shot_dist', 'mean'),
    avg_shot_angle=('avg_shot_angle', 'mean'),
    # Carry
    career_carries=('total_carries', 'sum'),
    career_progressive_carries=('progressive_carries', 'sum'),
    # Defensive
    career_def_actions=('total_def_actions', 'sum'),
    # Centrality (average across matches)
    avg_betweenness=('betweenness', 'mean'),
    avg_closeness=('closeness', 'mean'),
    avg_weighted_degree=('weighted_degree', 'mean'),
).reset_index()

# --- Correct conversion rates (goals/shots, shots_on_target/shots) ---
career_agg['goal_conversion_pct'] = np.where(
    career_agg['career_shots'] > 0,
    (career_agg['career_goals'] / career_agg['career_shots']) * 100,
    np.nan
)
career_agg['shot_accuracy_pct'] = np.where(
    career_agg['career_shots'] > 0,
    (career_agg['career_shots_on_target'] / career_agg['career_shots']) * 100,
    np.nan
)

# Merge career minutes from Phase 2 to compute career per‑90
career_summary_path = PATHS["processed"] / "statistical" / "player_summary.parquet"
if career_summary_path.exists():
    sum_df = pd.read_parquet(career_summary_path)
    min_col = 'Total_Minutes' if 'Total_Minutes' in sum_df.columns else 'career_minutes'
    career_min_df = sum_df[['canonical_id', min_col]].drop_duplicates()
    career_min_df.columns = ['canonical_id', 'career_minutes']

    career_agg = career_agg.merge(career_min_df, on='canonical_id', how='left')

    # Career per‑90
    for col in ['career_passes', 'career_progressive_passes', 'career_shots',
                'career_goals', 'career_xG', 'career_carries',
                'career_progressive_carries', 'career_def_actions']:
        career_agg[f'{col}_per90'] = np.where(
            career_agg['career_minutes'] > 0,
            career_agg[col] / (career_agg['career_minutes'] / 90),
            np.nan
        )
else:
    print("⚠️ Career minutes file not found; per‑90 career rates will be NaN.")

print(f"✅ Career spatial summaries built: {len(career_agg)} players.")


In [ ]:
# ============================================================
# Cell 10: Zone Occupancy (with Percentages) & Save Outputs
# ============================================================
import json

# Zone occupancy per player (all events with location)
zone_counts = events_spatial.groupby(['player', 'zone']).size().unstack(fill_value=0).reset_index()
zone_counts = zone_counts.rename(columns={'player': 'statsbomb_player'})
zone_counts['canonical_id'] = zone_counts['statsbomb_player'].map(statsbomb_to_canonical)
zone_counts['common_name'] = zone_counts['statsbomb_player'].map(statsbomb_to_common)

# Ensure all three zones exist
for z in ['defensive', 'midfield', 'attacking']:
    if z not in zone_counts.columns:
        zone_counts[z] = 0

zone_counts['total_touches'] = zone_counts[['defensive', 'midfield', 'attacking']].sum(axis=1)

# Zone percentages (useful for ML)
zone_counts['defensive_pct'] = zone_counts['defensive'] / zone_counts['total_touches'] * 100
zone_counts['midfield_pct'] = zone_counts['midfield'] / zone_counts['total_touches'] * 100
zone_counts['attacking_pct'] = zone_counts['attacking'] / zone_counts['total_touches'] * 100

# Pen‑area occupancy
pen_counts = events_spatial.groupby(['player', 'pen_zone']).size().unstack(fill_value=0).reset_index()
pen_counts = pen_counts.rename(columns={'player': 'statsbomb_player'})
pen_counts['canonical_id'] = pen_counts['statsbomb_player'].map(statsbomb_to_canonical)

# --- Save all spatial outputs ---
OUTPUT = PATHS["output_spatial"]
match_agg.to_parquet(OUTPUT / "player_match_spatial.parquet", index=False)
career_agg.to_parquet(OUTPUT / "player_spatial_summary.parquet", index=False)
zone_counts.to_parquet(OUTPUT / "player_zone_summary.parquet", index=False)
pen_counts.to_parquet(OUTPUT / "player_penalty_zone_summary.parquet", index=False)

# Save metadata manifest
meta = {
    "phase": "3_spatial_engine",
    "pipeline_version": "2.1",
    "seed": SEED,
    "generated_at": pd.Timestamp.now().isoformat(),
    "datasets": {
        "player_match_spatial": "player_match_spatial.parquet",
        "player_spatial_summary": "player_spatial_summary.parquet",
        "player_zone_summary": "player_zone_summary.parquet",
        "player_penalty_zone_summary": "player_penalty_zone_summary.parquet",
    },
    "features_produced": [
        "progressive_passes", "progressive_carries", "key_passes",
        "avg_shot_dist", "avg_shot_angle", "goal_conversion_pct", "shot_accuracy_pct",
        "passing_network_betweenness", "passing_network_closeness",
        "defensive_third_actions", "zone_occupancy_pct", "..."
    ],
    "notes": "Memory‑optimized: vectorized zone computation, safe .copy(), progressive thresholds: pass 30, carry 15. Centrality without pagerank."
}
with open(OUTPUT / "phase3_manifest.json", 'w') as f:
    json.dump(meta, f, indent=2)

print("💾 All spatial outputs saved with metadata.")


In [ ]:
# ============================================================
# Cell 11: Summary & Validation
# ============================================================
print("\n" + "="*70)
print("📊 PHASE 3 SPATIAL ENGINE — SUMMARY")
print("="*70)
print(f"Full events processed: {len(full_events)}")
print(f"Events with coordinates: {len(events_spatial)}")
print(f"Player‑match records: {len(match_agg)}")
print(f"Unique players: {match_agg['canonical_id'].nunique()}")
print(f"Unique matches: {match_agg['match_id'].nunique()}")
if not career_agg.empty:
    print(f"Career summaries: {len(career_agg)}")
    print("\nTop 3 by career progressive passes per 90:")
    if 'career_progressive_passes_per90' in career_agg.columns:
        display(career_agg[['common_name', 'career_progressive_passes_per90',
                            'avg_betweenness', 'avg_closeness', 'avg_shot_dist']]
                .sort_values('career_progressive_passes_per90', ascending=False).head(3))
print("\n✅ Phase 3 complete. Outputs in:", OUTPUT)

In [ ]:
# ============================================================
# Cell 12: Self‑Healing GitHub Sync (Phase 3 Backup)
# ============================================================
import os, subprocess
from pathlib import Path
from google.colab import userdata
from IPython.display import Javascript, display

REPO = Path("/content/drive/MyDrive/GOAT-Net")
os.chdir(REPO)

print("💾 Force saving notebook checkpoint...")
display(Javascript("IPython.notebook.save_checkpoint();"))

def git(*args):
    return subprocess.run(["git", *args], capture_output=True, text=True)

try:
    TOKEN = userdata.get("GITHUB_TOKEN")
    if not TOKEN: raise ValueError("❌ GITHUB_TOKEN not set in Colab secrets.")

    AUTH_URL = f"https://{TOKEN}@github.com/AtiX-Algo/GOAT-Net.git"
    CLEAN_URL = "https://github.com/AtiX-Algo/GOAT-Net.git"

    if not (REPO / ".git").exists():
        print("⚠️ .git folder missing. Re‑initializing...")
        git("init")
        git("remote", "add", "origin", CLEAN_URL)
        git("branch", "-M", "main")

    git("config", "user.name", "AtiX-Algo")
    git("config", "user.email", "atix.algo@gmail.com")
    git("remote", "set-url", "origin", AUTH_URL)

    git("add", "notebooks/")
    git("add", "data/processed/spatial/")
    if (REPO / "data" / "processed" / "spatial" / "phase3_manifest.json").exists():
        git("add", "data/processed/spatial/phase3_manifest.json")

    print("\n🔄 Files staged for commit:")
    print(git("status", "--short").stdout or "No changes staged.")

    commit = git("commit", "-m", "feat: complete Phase 3 spatial engine (v2.1 – memory optimized, safe copies)")
    if "nothing to commit" not in commit.stdout: print(commit.stdout)

    git("pull", "--no-rebase", "origin", "main")
    print("📤 Pushing Phase 3 to GitHub...")
    push = git("push", "-u", "origin", "main", "--force")
    git("remote", "set-url", "origin", CLEAN_URL)

    if push.returncode == 0:
        print("✅ Push successful! Spatial Engine is securely backed up.")
    else:
        print("❌ Push failed:\n", push.stderr.replace(TOKEN, "***HIDDEN***"))

except Exception as e:
    token_str = TOKEN if 'TOKEN' in locals() and TOKEN else "UNKNOWN_TOKEN"
    print(f"\n⚠️ GitHub Sync Skipped: {str(e).replace(token_str, '***HIDDEN***')}")

In [ ]:
# ============================================================
# GOAT-Net Safe GitHub Sync
# Syncs ALL notebooks, source code, metadata, and processed data
# ============================================================
import os, subprocess
from pathlib import Path
from google.colab import userdata
from IPython.display import display, Javascript

# Force Colab to save the notebook right before pushing
display(Javascript('IPython.notebook.save_checkpoint();'))

REPO = Path("/content/drive/MyDrive/GOAT-Net")

# Ensure repo exists
if not REPO.exists():
    raise FileNotFoundError(f"❌ Repository not found at {REPO}. Check your mount and path.")

os.chdir(REPO)

def run_git(args, **kwargs):
    """Run a git command and return the result."""
    return subprocess.run(["git"] + args, cwd=REPO, capture_output=True, text=True, **kwargs)

def print_git_error(cmd_name, result, token=None):
    """Print git errors safely without exposing tokens."""
    error_msg = result.stderr or result.stdout
    if token:
        error_msg = error_msg.replace(token, "***HIDDEN***")
    print(f"⚠️ {cmd_name} warning:\n{error_msg}")

try:
    TOKEN = userdata.get("GITHUB_TOKEN")
    if not TOKEN:
        raise ValueError("❌ GITHUB_TOKEN not set in Colab secrets.")

    REPO_SLUG = "AtiX-Algo/GOAT-Net"
    CLEAN_URL = f"https://github.com/{REPO_SLUG}.git"
    AUTH_URL = f"https://{TOKEN}@github.com/{REPO_SLUG}.git"

    # ============================================
    # 1. Repository Initialization (Safe)
    # ============================================
    if not (REPO / ".git").exists():
        print("🔧 Initializing git repository...")
        run_git(["init"])
        run_git(["remote", "add", "origin", CLEAN_URL])
        run_git(["branch", "-M", "main"])

    # Configure git identity
    run_git(["config", "user.name", "AtiX-Algo"])
    run_git(["config", "user.email", "atix.algo@gmail.com"])
    run_git(["remote", "set-url", "origin", AUTH_URL])

    # ============================================
    # 2. Ensure .gitignore exists
    # ============================================
    gitignore_path = REPO / ".gitignore"
    if not gitignore_path.exists():
        print("📝 Creating .gitignore...")
        gitignore_content = """# Python
__pycache__/
*.pyc
*.pyo
*.egg-info/
dist/
build/

# Jupyter/Colab
.ipynb_checkpoints/
*.ipynb_checkpoints/

# Large raw data files (keep small CSVs, ignore massive parquets)
data/raw/modern/statsbomb/statsbomb_full_match_events.parquet

# Environment and secrets
.env
*.log

# OS files
.DS_Store
Thumbs.db
"""
        gitignore_path.write_text(gitignore_content)

    # ============================================
    # 3. Stage ALL changes (let .gitignore decide)
    # ============================================
    print("📦 Staging all changes...")
    add_result = run_git(["add", "-A"])

    # Check what's being staged
    status_result = run_git(["status", "--short"])
    staged_files = status_result.stdout.strip()

    if staged_files:
        print("\n📋 Files to be committed:")
        print(staged_files)
    else:
        print("\n✨ No changes to commit. Everything is up to date!")
        # Clean up remote URL before exiting
        run_git(["remote", "set-url", "origin", CLEAN_URL])
        raise SystemExit

    # ============================================
    # 4. Commit changes
    # ============================================
    print("\n💾 Committing changes...")
    commit_result = run_git(["commit", "-m", "chore: sync notebooks, source, metadata, and processed data"])

    if commit_result.returncode != 0:
        # Check if it's just "nothing to commit" (safety check)
        if "nothing to commit" in commit_result.stdout or "nothing to commit" in commit_result.stderr:
            print("✨ Nothing to commit after all.")
            run_git(["remote", "set-url", "origin", CLEAN_URL])
            raise SystemExit
        else:
            print_git_error("Commit", commit_result, TOKEN)

    # ============================================
    # 5. Pull latest changes (safe merge)
    # ============================================
    print("\n🔽 Pulling latest changes from GitHub...")

    # First, fetch to see what's on remote
    fetch_result = run_git(["fetch", "origin", "main"])

    # Try a normal pull with merge (not rebase)
    pull_result = run_git(["pull", "origin", "main", "--no-rebase"])

    if pull_result.returncode != 0:
        # Check if there's actually a conflict or if remote is just ahead
        print("ℹ️ Pull had issues. Checking if we need to merge...")

        # Get the difference between local and remote
        run_git(["fetch", "origin"])
        diff_result = run_git(["rev-list", "--count", "HEAD..origin/main"])

        if diff_result.stdout.strip() == "0" or not diff_result.stdout.strip():
            print("✅ Local is up to date with remote. Continuing with push.")
        else:
            print("ℹ️ Attempting merge...")
            merge_result = run_git(["merge", "origin/main", "--no-edit"])
            if merge_result.returncode != 0:
                print("⚠️ Manual merge required. Please resolve conflicts in GitHub.")
                print("   Your changes are committed locally and won't be lost.")
                print(f"   Conflicts: {merge_result.stderr.replace(TOKEN, '***HIDDEN***')}")
                run_git(["remote", "set-url", "origin", CLEAN_URL])
                exit(1)

    # ============================================
    # 6. Push to GitHub (NO force push)
    # ============================================
    print("\n📤 Pushing to GitHub...")
    push_result = run_git(["push", "origin", "main"])  # No --force!

    if push_result.returncode == 0:
        print("✅ Push successful! All your work is securely backed up to GitHub.")
        print(f"   Repository: https://github.com/{REPO_SLUG}")
    elif "non-fast-forward" in push_result.stderr:
        print("⚠️ Push rejected: Remote has changes you don't have locally.")
        print("   This is a SAFETY FEATURE protecting your remote work.")
        print("   Your changes are committed locally.")
        print("   To fix: Run this sync script again to pull and merge first.")
    else:
        print_git_error("Push", push_result, TOKEN)
        print("ℹ️ Your changes are committed locally but couldn't be pushed.")

except Exception as e:
    # Safely print error without exposing token
    error_msg = str(e)
    if 'TOKEN' in locals() and TOKEN:
        error_msg = error_msg.replace(TOKEN, "***HIDDEN***")
    print(f"\n⚠️ GitHub Sync encountered an issue: {error_msg}")
    print("ℹ️ Your notebook and local changes are not affected.")

finally:
    # ============================================
    # ALWAYS clean up: Remove token from remote URL
    # ============================================
    try:
        if (REPO / ".git").exists():
            run_git(["remote", "set-url", "origin", CLEAN_URL])
            print("🔒 Remote URL secured (token removed).")
    except:
        pass  # Silent fail for cleanup